In [ ]:
# step 1
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
# step 2
from sklearn.model_selection import train_test_split
# strp 3
from xgboost import XGBClassifier

from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score

#### 데이터 불러오기 및 전처리

In [3]:
# 데이터 불러오기
top05 = pd.read_csv('./result/상위5개컬럼모음.csv', encoding='utf-8-sig')
top05.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2400000 entries, 0 to 2399999
Data columns (total 6 columns):
 #   Column         Dtype 
---  ------         ----- 
 0   Segment        object
 1   _2순위카드이용금액     int64 
 2   쇼핑_도소매_이용금액    int64 
 3   _1순위교통업종_이용금액  int64 
 4   연체입금원금_B0M     int64 
 5   잔액_일시불_B0M     int64 
dtypes: int64(5), object(1)
memory usage: 109.9+ MB


In [4]:
# 입력 / 타겟 분리
X = top05.drop('Segment', axis=1)
y = top05['Segment']

In [5]:
# 라벨 인코딩 (A~E → 0~4)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [6]:
# 표준화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#### 학습/검증 데이터 분할

In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

In [15]:
type(X_train)

numpy.ndarray

### XGBoost 모델 학습

In [11]:
import xgboost
print(xgboost.__version__)

3.0.2


In [ ]:
# 모델 정의 및 전체 학습
model = XGBClassifier(
    objective='multi:softmax',
    num_class=5,
    eval_metric='mlogloss',
    use_label_encoder=False,
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

model.fit(X_train, y_train)

c:\Users\COTTA\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [09:48:15] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None, num_class=5, ...)

##### Kfold 교차검증 5번

In [24]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold = 1

for train_idx, val_idx in skf.split(X_scaled, y_encoded):
    X_tr, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_tr, y_val = y_encoded[train_idx], y_encoded[val_idx]

    model = XGBClassifier(
        objective='multi:softmax',
        num_class=5,
        eval_metric='mlogloss',
        use_label_encoder=False,
        n_estimators=300,
        learning_rate=0.1,
        max_depth=6,
        random_state=fold
    )
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_val)

    # classification report 출력
    print(f"\n Fold {fold} classification report")
    print(classification_report(y_val, y_pred))

    # 모델 저장
    model.save_model(f"model/xgb_fold{fold}.json")

    fold += 1

c:\Users\COTTA\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:17:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



 Fold 1 classification report
              precision    recall  f1-score   support

           0       0.50      0.01      0.02       195
           1       0.00      0.00      0.00        28
           2       0.56      0.34      0.42     25518
           3       0.52      0.30      0.38     69849
           4       0.87      0.97      0.92    384410

    accuracy                           0.84    480000
   macro avg       0.49      0.32      0.35    480000
weighted avg       0.81      0.84      0.81    480000



c:\Users\COTTA\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:18:27] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



 Fold 2 classification report
              precision    recall  f1-score   support

           0       0.45      0.03      0.05       194
           1       0.50      0.03      0.06        29
           2       0.56      0.33      0.42     25518
           3       0.52      0.30      0.38     69849
           4       0.87      0.97      0.92    384410

    accuracy                           0.84    480000
   macro avg       0.58      0.33      0.37    480000
weighted avg       0.81      0.84      0.81    480000



c:\Users\COTTA\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:19:45] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



 Fold 3 classification report
              precision    recall  f1-score   support

           0       0.33      0.01      0.02       194
           1       0.00      0.00      0.00        29
           2       0.57      0.34      0.42     25518
           3       0.52      0.30      0.38     69848
           4       0.87      0.97      0.92    384411

    accuracy                           0.84    480000
   macro avg       0.46      0.32      0.35    480000
weighted avg       0.81      0.84      0.81    480000



c:\Users\COTTA\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:21:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



 Fold 4 classification report
              precision    recall  f1-score   support

           0       0.50      0.01      0.02       194
           1       0.00      0.00      0.00        29
           2       0.56      0.34      0.42     25518
           3       0.53      0.30      0.38     69848
           4       0.87      0.97      0.92    384411

    accuracy                           0.84    480000
   macro avg       0.49      0.32      0.35    480000
weighted avg       0.81      0.84      0.81    480000



c:\Users\COTTA\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:22:18] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



 Fold 5 classification report
              precision    recall  f1-score   support

           0       0.57      0.02      0.04       195
           1       0.00      0.00      0.00        29
           2       0.56      0.33      0.42     25518
           3       0.52      0.30      0.38     69848
           4       0.87      0.97      0.92    384410

    accuracy                           0.84    480000
   macro avg       0.51      0.32      0.35    480000
weighted avg       0.81      0.84      0.81    480000



In [25]:
# 교차검증 및 평균 기록용
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
macro_f1_list, weighted_f1_list, accuracy_list = [], [], []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, y_encoded), 1):
    X_tr, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_tr, y_val = y_encoded[train_idx], y_encoded[val_idx]

    model = XGBClassifier(
        objective='multi:softmax',
        num_class=5,
        eval_metric='mlogloss',
        use_label_encoder=False,
        n_estimators=300,
        learning_rate=0.1,
        max_depth=6,
        random_state=fold
    )
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_val)

    report = classification_report(y_val, y_pred, output_dict=True)

    # 각 지표 수집
    macro_f1_list.append(report['macro avg']['f1-score'])
    weighted_f1_list.append(report['weighted avg']['f1-score'])
    accuracy_list.append(report['accuracy'])

# 평균 출력
print("\n평균 성능 요약:")
print(f"Macro F1 평균:     {np.mean(macro_f1_list):.4f}")
print(f"Weighted F1 평균:  {np.mean(weighted_f1_list):.4f}")
print(f"Accuracy 평균:     {np.mean(accuracy_list):.4f}")

c:\Users\COTTA\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:32:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\COTTA\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:34:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\COTTA\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:35:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\COTTA\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:36:48] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtr


평균 성능 요약:
Macro F1 평균:     0.3526
Weighted F1 평균:  0.8132
Accuracy 평균:     0.8354


#### 모델 및 전처리 도구 저장

In [ ]:
import joblib
import os

# 저장 폴더 생성
os.makedirs("saved_models", exist_ok=True)

# 모델 저장
model.save_model("saved_models/xgb_segment_model.json")

# 인코더, 스케일러도 같이 저장
joblib.dump(le, "saved_models/label_encoder.pkl")
joblib.dump(scaler, "saved_models/scaler.pkl")

print("모델 및 전처리 도구 저장 완료")

#### 전체 데이터로 재학습

In [ ]:
# 전체 데이터로 다시 학습
model.fit(X_scaled, y_encoded)

# 최종 모델 저장
model.save_model("saved_models/xgb_segment_model_final.json")
print("전체 데이터로 재학습된 모델 저장 완료")